# Zehirli / Zehirsiz Böcek Veri Seti — Ek Veri (Benzer Görünümlü Çiftler)

İlk veri setinde (*Zehirli Bocek Veri Toplama*) bazı görünüm tiplerinin yalnızca zehirli üyeleri vardı. Örneğin tüm tırtıllar zehirliydi. Bu durumda model zehri değil şekli öğrenebilir ("tırtılsa zehirlidir"). Bu hataya **kestirme öğrenme (shortcut learning)** denir.

Bunu önlemek için bu notebook, **aynı aile / alt aile içinden** zehirli ve zehirsiz akrabaları ve **benzer görünümlü taklitçileri** ekler:

| Aile / görünüm tipi | Zehirli | Zehirsiz akraba / taklitçi |
|---|---|---|
| Apidae (arılar) | Bal arısı, bombus *(ilk veri)* | İğnesiz arılar (Meliponini) |
| Formicidae (karıncalar) | Ateş karıncası *(ilk veri)*, hasatçı karıncalar (*Pogonomyrmex*) | İğnesiz karıncalar (Formicinae) |
| Staphylinidae (kısakanatlılar) | *Paederus* *(ilk veri)* | Diğer kısakanatlılar (*Paederus* hariç) |
| Notodontidae (tırtıl) | Keseböceği tırtılı *(ilk veri)* | Diğer Notodontidae tırtılları (Thaumetopoeinae hariç) |
| Saturniidae (tırtıl) | Dikenli Hemileucinae tırtılları (*Automeris*, *Lonomia*...) | Zararsız Saturniinae tırtılları |
| Erebidae (tırtıl) | Kaşındıran kıllı Lymantriinae tırtılları | — |
| Tırtıllar (genel) | Kedi / sülük tırtılları *(ilk veri)* | Atmaca güvesi tırtılları (Sphingidae) |
| Arı / eşek arısı görünümü | Vespidae, Pompilidae *(ilk veri)* | Testereli arılar (Tenthredinidae), arı taklitçi güveler (Sesiidae) |
| Yarımkanatlılar | Suikastçı böcekler *(ilk veri)* | Pis kokulu böcekler (Pentatomidae), Coreidae |
| Kınkanatlılar | Kabarcık böcekleri *(ilk veri)* | Yaprak böcekleri (Chrysomelidae) |

Kaynak, filtreler ve yöntem ilk notebook ile aynıdır: iNaturalist **Research Grade** gözlemleri, **CC0 / CC-BY / CC-BY-NC** lisanslı fotoğraflar, gözlem başına 1 fotoğraf, grup başına en fazla 2.000 fotoğraf. Tırtıl gruplarında yalnızca **larva** evresi alınır.

*Not:* Aynı türün zehirsiz bireyi de vardır. Erkek arılar ve eşek arıları iğnesizdir. Ancak fotoğrafta dişilerden güvenilir şekilde ayırt edilemedikleri için veri setine alınmadı.

## 1. Ayarlar ve takson grupları

In [ ]:
import io, os, time
import requests
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

API = "https://api.inaturalist.org/v1"
LISANSLAR = ["cc0", "cc-by", "cc-by-nc"]
MAKS_FOTO = 2000      # grup başına en fazla fotoğraf
MAKS_KENAR = 400      # fotoğraflar en uzun kenarı 400 piksel olacak şekilde küçültülür
KOK = "/kaggle/working/zehirli_bocek_ek_veri"
LARVA = {"term_id": 1, "term_value_id": 6}   # iNaturalist "Life Stage = Larva" etiketi

GRUPLAR = [
    # takson,          sınıf,      larva, hariç tutulan takson, eşleştiği zehirli grup, açıklama
    ("Pogonomyrmex",   "zehirli",  False, None,              "Formicinae",   "Hasatçı karıncalar (güçlü iğne)"),
    ("Hemileucinae",   "zehirli",  True,  None,              "Saturniinae",  "Dikenli ipek güvesi tırtılları"),
    ("Lymantriinae",   "zehirli",  True,  None,              "Sphingidae",   "Kaşındıran kıllı tırtıllar"),
    ("Meliponini",     "zehirsiz", False, None,              "Apis mellifera / Bombus", "İğnesiz arılar"),
    ("Formicinae",     "zehirsiz", False, None,              "Solenopsis / Pogonomyrmex", "İğnesiz karıncalar"),
    ("Staphylinidae",  "zehirsiz", False, "Paederus",        "Paederus",     "Paederus dışındaki kısakanatlılar"),
    ("Notodontidae",   "zehirsiz", True,  "Thaumetopoeinae", "Thaumetopoea", "Keseböceği dışındaki Notodontidae tırtılları"),
    ("Saturniinae",    "zehirsiz", True,  None,              "Hemileucinae", "Zararsız ipek güvesi tırtılları"),
    ("Sphingidae",     "zehirsiz", True,  None,              "Megalopygidae / Limacodidae / Lymantriinae", "Atmaca güvesi tırtılları"),
    ("Tenthredinidae", "zehirsiz", False, None,              "Vespidae / Pompilidae", "Testereli arılar (iğnesiz)"),
    ("Sesiidae",       "zehirsiz", False, None,              "Vespidae / Apis mellifera", "Arı taklitçi güveler"),
    ("Pentatomidae",   "zehirsiz", False, None,              "Reduviidae",   "Pis kokulu böcekler"),
    ("Coreidae",       "zehirsiz", False, None,              "Reduviidae",   "Coreidae (yarımkanatlı)"),
    ("Chrysomelidae",  "zehirsiz", False, None,              "Meloidae",     "Yaprak böcekleri"),
]
gruplar = pd.DataFrame(GRUPLAR, columns=["takson", "sinif", "larva", "haric", "esi", "aciklama"])
gruplar["grup"] = gruplar.takson.str.replace(" ", "_")
gruplar

## 2. iNaturalist API yardımcıları ve gözlem sayıları

In [ ]:
oturum = requests.Session()
oturum.headers["User-Agent"] = "zehirli-bocek-ders-projesi (Kaggle notebook)"

def api(yol, **params):
    """iNaturalist API isteği; istek sınırına uymak için her istek arasında 1 sn bekler."""
    for deneme in range(5):
        time.sleep(1.0)
        try:
            r = oturum.get(f"{API}/{yol}", params=params, timeout=60)
            if r.status_code == 200:
                return r.json()
        except requests.RequestException:
            pass
        time.sleep(5 * (deneme + 1))
    raise RuntimeError(f"API isteği başarısız: {yol} {params}")

def takson_bul(ad):
    sonuclar = api("taxa", q=ad, is_active="true", locale="tr", per_page=30)["results"]
    return next(t for t in sonuclar if t["name"] == ad)

def filtre(g):
    p = dict(taxon_id=g.taxon_id, quality_grade="research", photos="true",
             photo_license=",".join(LISANSLAR))
    if pd.notna(g.haric_id):
        p["without_taxon_id"] = int(g.haric_id)
    if g.larva:
        p.update(LARVA)
    return p

taksonlar = [takson_bul(ad) for ad in gruplar.takson]
gruplar["taxon_id"] = [t["id"] for t in taksonlar]
gruplar["rank"] = [t["rank"] for t in taksonlar]
gruplar["ad_tr"] = [t.get("preferred_common_name", "") for t in taksonlar]
gruplar["haric_id"] = [takson_bul(h)["id"] if h else None for h in gruplar.haric]
gruplar["gozlem"] = [api("observations", **filtre(g), per_page=0)["total_results"] for g in gruplar.itertuples()]
gruplar["alinacak"] = gruplar.gozlem.clip(upper=MAKS_FOTO)
print(gruplar.groupby("sinif").alinacak.sum(), "\nTOPLAM:", gruplar.alinacak.sum())
gruplar[["takson", "sinif", "rank", "larva", "haric", "esi", "gozlem", "alinacak"]]

## 3. Fotoğraf listesi ve kaynak bilgileri (metadata)

In [ ]:
def gozlemleri_cek(g):
    satirlar, sayfa = [], 1
    while len(satirlar) < MAKS_FOTO:
        sonuc = api("observations", **filtre(g), per_page=200, page=sayfa,
                    order_by="id", order="desc")["results"]
        if not sonuc:
            break
        for o in sonuc:
            foto = next((p for p in o["photos"] if p.get("license_code") in LISANSLAR), None)
            if foto is None:
                continue
            satirlar.append(dict(
                sinif=g.sinif, grup=g.grup, obs_id=o["id"], photo_id=foto["id"],
                takson=o["taxon"]["name"], takson_rank=o["taxon"]["rank"],
                lisans=foto["license_code"], atif=foto.get("attribution"),
                kullanici=o["user"]["login"], tarih=o.get("observed_on"),
                yer=o.get("place_guess"),
                url=foto["url"].replace("/square.", "/medium."),
            ))
        sayfa += 1
    return satirlar[:MAKS_FOTO]

satirlar = []
for g in gruplar.itertuples():
    satirlar += gozlemleri_cek(g)
    print(f"{g.grup:15s} {g.sinif:9s} toplam liste: {len(satirlar)}")

meta = pd.DataFrame(satirlar).drop_duplicates("photo_id")
meta["dosya"] = ("images/" + meta.sinif + "/" + meta.grup + "/" + meta.photo_id.astype(str) + ".jpg")
print(meta.shape, "| farklı takson:", meta.takson.nunique())
meta.lisans.value_counts()

## 4. Fotoğrafları indirme ve küçültme

In [ ]:
def indir(url, hedef):
    if os.path.exists(hedef):
        return True
    for deneme in range(3):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                img = Image.open(io.BytesIO(r.content)).convert("RGB")
                img.thumbnail((MAKS_KENAR, MAKS_KENAR))
                img.save(hedef, "JPEG", quality=90)
                return True
        except Exception:
            pass
        time.sleep(2 * (deneme + 1))
    return False

for d in meta.dosya.map(lambda p: os.path.dirname(os.path.join(KOK, p))).unique():
    os.makedirs(d, exist_ok=True)

t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    isler = {ex.submit(indir, u, os.path.join(KOK, p)): p for u, p in zip(meta.url, meta.dosya)}
    basarili = {}
    for i, f in enumerate(as_completed(isler), 1):
        basarili[isler[f]] = f.result()
        if i % 2000 == 0:
            print(f"{i}/{len(isler)} indirildi ({(time.time() - t0) / 60:.1f} dk)", flush=True)

meta["indirildi"] = meta.dosya.map(basarili)
meta = meta[meta.indirildi].drop(columns="indirildi")
boyut = sum(os.path.getsize(os.path.join(KOK, p)) for p in meta.dosya) / 1e9
print(f"İndirilen: {len(meta)} | Başarısız: {len(basarili) - len(meta)} | "
      f"Süre: {(time.time() - t0) / 60:.1f} dk | Boyut: {boyut:.2f} GB")

## 5. Kayıt dosyaları ve özet

In [ ]:
meta.to_csv(f"{KOK}/metadata.csv", index=False)
gruplar.to_csv(f"{KOK}/gruplar.csv", index=False)

lisans_sayilari = meta.lisans.value_counts().to_dict()
with open(f"{KOK}/KAYNAK.md", "w", encoding="utf-8") as f:
    f.write("# Kaynak ve Lisans\n\n")
    f.write("Fotoğraflar iNaturalist (https://www.inaturalist.org) Research Grade gözlemlerinden alınmıştır.\n")
    f.write(f"Toplam {len(meta)} fotoğraf, {meta.takson.nunique()} farklı takson.\n")
    f.write(f"Lisanslar: {lisans_sayilari}\n")
    f.write("Her fotoğrafın fotoğrafçısı ve lisansı metadata.csv dosyasının 'atif' ve 'lisans' sütunlarındadır.\n")

ozet = meta.groupby(["sinif", "grup"]).size().rename("foto").reset_index()
print(meta.sinif.value_counts())
renk = ozet.sinif.map({"zehirli": "tab:red", "zehirsiz": "tab:green"})
plt.figure(figsize=(12, 4))
plt.bar(ozet.grup, ozet.foto, color=renk)
plt.xticks(rotation=60, ha="right"); plt.ylabel("Fotoğraf sayısı")
plt.title("Ek veri: gruplara göre fotoğraf sayısı (kırmızı: zehirli, yeşil: zehirsiz)")
plt.tight_layout(); plt.show()

In [ ]:
ornekler = meta.groupby("grup").head(2).reset_index(drop=True)
fig, axes = plt.subplots(4, 7, figsize=(20, 12))
for ax, s in zip(axes.flat, ornekler.itertuples()):
    ax.imshow(Image.open(os.path.join(KOK, s.dosya)))
    ax.set_title(f"{s.grup}\n({s.sinif})", fontsize=8, color="red" if s.sinif == "zehirli" else "green")
    ax.axis("off")
plt.tight_layout(); plt.show()